In [ ]:
# Cell 1: Complete setup (run this first)
print("Starting setup...")

# Clone repository
!git clone https://github.com/PurplePegasus08/ANPD-using-YOLO.git
%cd ANPD-using-YOLO

# Install ALL required packages
print("\nInstalling ultralytics...")
!pip install ultralytics -q

print("Installing OpenCV...")
!pip install opencv-python -q

print("Installing EasyOCR...")
!pip install easyocr -q

print("Installing matplotlib...")
!pip install matplotlib -q

print("Installing langchain components...")
!pip install langchain langchain-text-splitters langchain-community -q

print("✅ All packages installed successfully!")

# Verify installations
import importlib
packages = ['ultralytics', 'cv2', 'easyocr', 'matplotlib']
for pkg in packages:
    try:
        if pkg == 'cv2':
            import cv2
        else:
            importlib.import_module(pkg)
        print(f"✓ {pkg} is available")
    except ImportError:
        print(f"✗ {pkg} is missing")

In [ ]:
# Cell 2: Download the pre-trained model
!wget https://github.com/PurplePegasus08/ANPD-using-YOLO/raw/main/best.pt -O best.pt

# Verify download
import os
if os.path.exists('best.pt'):
    size = os.path.getsize('best.pt') / (1024*1024)
    print(f"✅ Model downloaded successfully! Size: {size:.2f} MB")
else:
    print("❌ Model download failed")

In [ ]:
# YOLO 5 and easy OCR 
!pip install ultralytics easyocr -q

from ultralytics import YOLO
import cv2
import easyocr
from google.colab import files
import numpy as np
import os

def enhance_plate_for_ocr(plate_img):
    """Enhance plate image for better OCR"""
    # Convert to grayscale
    if len(plate_img.shape) == 3:
        gray = cv2.cvtColor(plate_img, cv2.COLOR_RGB2GRAY)
    else:
        gray = plate_img
    
    # Resize to make it larger
    gray = cv2.resize(gray, None, fx=2, fy=2, interpolation=cv2.INTER_CUBIC)
    
    # Apply contrast enhancement
    clahe = cv2.createCLAHE(clipLimit=2.5, tileGridSize=(8,8))
    enhanced = clahe.apply(gray)
    
    # Denoise
    denoised = cv2.medianBlur(enhanced, 3)
    
    # Binarize
    _, binary = cv2.threshold(denoised, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
    
    return binary

# Load the working plate detector (Persian model - we know it works)
print("="*50)
print("Loading License Plate Detector...")
detector = YOLO('working_model.pt')  # This detects plates, not cars

print("Loading EasyOCR for text reading...")
reader = easyocr.Reader(['en'])
print("✅ Models loaded!")
print("="*50)

print("\n📁 Upload your LTO plate image:")
uploaded = files.upload()

for img_name, img_data in uploaded.items():
    print(f"\n{'─'*50}")
    print(f"Processing: {img_name}")
    print(f"{'─'*50}")
    
    # Save image
    with open(img_name, 'wb') as f:
        f.write(img_data)
    
    # Read image
    img = cv2.imread(img_name)
    img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    
    # Detect plates (using the Persian model that found plates before)
    results = detector(img_rgb, conf=0.3)
    
    if results[0].boxes and len(results[0].boxes) > 0:
        print(f"✓ Found {len(results[0].boxes)} potential plate region(s)")
        
        # Process all detections and collect unique plates
        all_plate_texts = []
        
        for box in results[0].boxes:
            confidence = float(box.conf[0])
            x1, y1, x2, y2 = map(int, box.xyxy[0].tolist())
            
            # Crop the plate region
            plate_crop = img_rgb[y1:y2, x1:x2]
            
            if plate_crop.size > 0:
                # Enhance for OCR
                enhanced_plate = enhance_plate_for_ocr(plate_crop)
                
                # Read text with EasyOCR
                ocr_result = reader.readtext(enhanced_plate, detail=0, paragraph=False)
                
                if ocr_result:
                    plate_text = ocr_result[0].strip()
                    # Only keep reasonable length texts (likely plates)
                    if len(plate_text) >= 3:
                        all_plate_texts.append({
                            'text': plate_text,
                            'confidence': confidence,
                            'bbox': (x1, y1, x2, y2)
                        })
        
        # Remove duplicates and get the most complete plate
        if all_plate_texts:
            # Sort by text length (longest likely most complete)
            all_plate_texts.sort(key=lambda x: len(x['text']), reverse=True)
            
            # Get unique plates
            unique_plates = []
            for plate in all_plate_texts:
                if plate['text'] not in [p['text'] for p in unique_plates]:
                    unique_plates.append(plate)
            
            print("\n" + "="*50)
            print("📋 LICENSE PLATE DETECTION RESULTS")
            print("="*50)
            
            for i, plate in enumerate(unique_plates, 1):
                print(f"\n  🚗 PLATE #{i}")
                print(f"     📝 NUMBER: {plate['text']}")
                print(f"     🎯 Confidence: {plate['confidence']:.2f}")
            
            print("\n" + "="*50)
            
            # Save cropped plate for inspection
            best_plate = unique_plates[0]
            x1, y1, x2, y2 = best_plate['bbox']
            plate_crop = img_rgb[y1:y2, x1:x2]
            cv2.imwrite(f"cropped_plate_{img_name}", cv2.cvtColor(plate_crop, cv2.COLOR_RGB2BGR))
            print(f"\n💾 Cropped plate saved as: cropped_plate_{img_name}")
            
        else:
            print("\n⚠️ No valid plate text could be read")
            print("   The detector found regions but OCR couldn't extract text")
            
    else:
        print("❌ No license plate detected in the image")
        print("\n   💡 Tips for better results:")
        print("      - Ensure the license plate is clearly visible")
        print("      - Good lighting is essential")
        print("      - Plate should be facing the camera")
        print("      - Try a higher resolution image")
    
    # Cleanup
    os.remove(img_name)

print("\n" + "="*50)
print("✨ Processing complete!")
print("="*50)